# 05 — Spectral Indices per LCZ Class

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ByMaxAnjos/LCZ4py/blob/main/notebooks/general/05_spectral_indices.en.ipynb)

**Learning objective.** By the end of this notebook you will download a small Sentinel-2 scene over a city via the Microsoft Planetary Computer, compute a battery of spectral indices (NDVI, NDBI, and friends) from its bands, and summarize how those indices vary across Local Climate Zone (LCZ) classes — with rigorous effect-size statistics, not just eyeballed boxplots.

## Why spectral indices matter for LCZ analysis

The Local Climate Zone scheme (Stewart & Oke, 2012) classifies urban and natural landscapes into 17
standardized classes based on their surface structure (building/tree height, spacing, imperviousness)
and land cover. It is a powerful *morphological* classification — but morphology alone doesn't tell
you what's actually growing, built, or bare on the ground in a given zone. Two cities' "LCZ 6 — open
lowrise" neighborhoods can look identical in a 3D model yet have wildly different amounts of street
trees, lawns, or paved surface, and that difference matters enormously for how each zone actually
behaves thermally.

This is where spectral indices come in. Computed pixel-by-pixel from satellite surface reflectance,
they give a direct, physically grounded read of land-cover composition:

- **NDVI** (Normalized Difference Vegetation Index) — the standard proxy for green vegetation
  fraction and canopy vigor. Vegetation strongly reflects near-infrared (NIR) light and absorbs red
  light for photosynthesis, so `(NIR - Red) / (NIR + Red)` is high over healthy canopy and low/negative
  over water, bare soil, or impervious surfaces.
- **NDBI** (Normalized Difference Built-up Index) — the complementary proxy for impervious/built-up
  surface fraction, exploiting the fact that built materials reflect more shortwave-infrared (SWIR)
  than NIR, unlike vegetation.
- Water indices (NDWI, MNDWI), soil indices (BSI), and more — each isolates a different component of
  the land-cover mosaic that a single LCZ code compresses into one category.

Why does this matter for urban climate science? Because vegetation fraction and impervious fraction
are two of the *primary physical drivers* of the Urban Heat Island effect that the LCZ scheme was
designed to help explain in the first place (Stewart & Oke, 2012; Anjos, M. et al., 2025, *Scientific
Reports*, https://www.nature.com/articles/s41598-025-92000-0). NDVI and NDBI per LCZ class let you go
from "LCZ 2 (compact midrise) and LCZ 6 (open lowrise) are structurally different" to "LCZ 2 has
measurably less vegetation and more built-up surface than LCZ 6, with effect size d = X" — a
quantitative, falsifiable, and comparable-across-cities statement. Cohen's d, used throughout this
notebook, expresses the difference between two groups' means in units of pooled standard deviation:
by convention, |d| < 0.2 is negligible, 0.2-0.5 is small, 0.5-0.8 is medium, and > 0.8 is a large,
practically important difference. That is a far more defensible basis for policy or design
recommendations ("prioritize LCZ 2 blocks for green-infrastructure retrofits") than a visual
comparison of boxplots.

This notebook builds the full pipeline: `lcz_get_map` → `lcz_get_planetary_computer` (Sentinel-2
scene) → `lcz_get_indices` (spectral indices) → `lcz_cal_indices` (per-LCZ statistics and charts). It
also shows `lcz_cal_indexes`, a more general zonal-statistics tool (covered fully in notebook 07) that
can summarize *any* gridded variable by LCZ class — reusing an NDVI band as a quick example of why you
don't need a dedicated function every time you want a per-class summary of a raster.

In [1]:
!pip -q install "LCZ4py[all] @ git+https://github.com/ByMaxAnjos/LCZ4py.git"


[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: /Applications/QGIS.app/Contents/MacOS/bin/python3 -m pip install --upgrade pip
ERROR: Package 'lcz4py' requires a different Python: 3.9.5 not in '>=3.10'


## 1. Get the LCZ map

`lcz_get_map(city=...)` geocodes the city, downloads only the intersecting window of the global LCZ
raster (Demuzere et al., 2022) from Zenodo, and caches both the city boundary and the clipped GeoTIFF
locally. We use **Juiz de Fora** (Liechtenstein's capital) — a tiny city, so the Sentinel-2 download in the
next step stays fast.

In [2]:
from LCZ4py import lcz_get_map

lcz_map_path = lcz_get_map(city="Juiz de Fora")
print(lcz_map_path)

13:37:33 - LCZ4py._internal._lcz_map_engine - INFO - Streaming COG and clipping to geometry...


13:37:39 - rasterio._env - WARNING - CPLE_IllegalArg in tmpl1v6ssw0.tif: BLOCKXSIZE can only be used with TILED=YES


13:37:39 - LCZ4py._internal._lcz_map_engine - INFO - Saved to clipped cache.


/Users/co2map/.lcz4r_cache/clipped_c7031cb44f08aaf6.tif


The returned path is a GeoTIFF with one band of LCZ class codes (1-17), clipped to Juiz de Fora's administrative boundary. Every downstream step in this notebook uses this same raster as the reference grid — indices and gridded variables all get reprojected/cropped onto it.

## 2. Download a Sentinel-2 scene via the Planetary Computer

`lcz_get_planetary_computer` streams Microsoft Planetary Computer STAC assets cropped to the LCZ
map's footprint — no manual authentication needed (Planetary Computer's public collections are
keyless). Key parameters:

- `collection="sentinel-2-l2a"` — atmospherically-corrected surface reflectance, ~10-60 m resolution.
- `assets` — which bands to fetch. The built-in shortcut only grabs B04/B03/B02/B08 (red/green/blue/
  NIR); we add `"B11"` (SWIR1) explicitly so `lcz_get_indices` can also compute NDBI and MNDWI, not
  just NDVI-family indices.
- `start_date`/`end_date` — a short window keeps the STAC search and download fast.
- `max_cloud_cover` — filters out cloudy scenes.
- `max_items` — how many STAC items (scenes) to mosaic/composite. We keep this small (2) for a quick,
  Colab-friendly download; increase it for a more complete cloud-free composite in real work.

This cell can be slow depending on Planetary Computer's load and network conditions — that's expected
for a real satellite download, even a small one.

In [3]:
from LCZ4py import lcz_get_planetary_computer

s2 = lcz_get_planetary_computer(
    lcz_map_path,
    collection="sentinel-2-l2a",
    assets=["B04", "B03", "B02", "B08", "B11"],  # +B11 (SWIR1) unlocks NDBI/MNDWI
    start_date="2023-06-01",
    end_date="2023-08-31",
    max_cloud_cover=30.0,
    max_items=2,
)
print(s2.path)
print(s2.bands)

13:37:39 - LCZ4py.general.lcz_get_planetary_computer - INFO - Using cached Planetary Computer stack: /Users/co2map/.lcz4r_cache/pc_sentinel-2-l2a_da71897902d6752c.tif


/Users/co2map/.lcz4r_cache/pc_sentinel-2-l2a_da71897902d6752c.tif
['B04', 'B03', 'B02', 'B08', 'B11']


`s2` is an `LCZPCResult` whose `.path` points to a multi-band GeoTIFF (one band per requested asset), already cropped and pixel-aligned to the LCZ map's grid, and whose `.bands` lists the asset names in band order. Because `cache=True` is the default, this GeoTIFF is persisted on disk — required for `lcz_get_indices`, which re-reads from disk rather than trusting only the in-memory array.

## 3. Compute spectral indices

`lcz_get_indices` reads the band stack, maps sensor-specific band names (`B04`, `B08`, ...) onto
canonical roles (red, nir, swir1, ...) automatically, and computes every index whose required bands
are present — or a restricted `indices=[...]` list. With B04/B03/B02/B08/B11 available we can compute
the full NDVI family plus NDBI, MNDWI, and more (anything needing SWIR2 or a thermal band is
unavailable here — Sentinel-2 has no thermal band, and we didn't fetch B12).

We restrict the request to four widely-used indices for a focused analysis:

- **NDVI** — vegetation fraction/vigor (vegetation family).
- **NDBI** — built-up/impervious fraction (urban family).
- **MNDWI** — open water fraction (water family, using SWIR1 — more robust against built-up
  false-positives than plain NDWI).
- **EVI** — enhanced vegetation index, corrects some of NDVI's atmospheric and soil-background
  sensitivity, useful as a cross-check on NDVI in dense canopy.

In [4]:
from LCZ4py import lcz_get_indices

idx = lcz_get_indices(s2, indices=["NDVI", "NDBI", "MNDWI", "EVI"])
print(idx.indices)
print(idx.array.shape)  # (n_indices, H, W)

13:37:39 - LCZ4py.general.lcz_get_indices - INFO - Computed {val NDVI}.


13:37:39 - LCZ4py.general.lcz_get_indices - INFO - Computed {val NDBI}.


13:37:39 - LCZ4py.general.lcz_get_indices - INFO - Computed {val MNDWI}.


13:37:39 - LCZ4py.general.lcz_get_indices - INFO - Computed {val EVI}.


['NDVI', 'NDBI', 'MNDWI', 'EVI']
(4, 534, 601)


`idx.array` is a `(4, H, W)` float32 stack, one band per requested index, NaN wherever the source Sentinel-2 stack was NaN (already limited to Juiz de Fora's footprint). `idx.indices` gives the band order. At this point every pixel has a vegetation score, a built-up score, and a water score — but they're not yet related to LCZ class. That's `lcz_cal_indices`'s job.

## 4. Per-LCZ-class statistics: box plots

`lcz_cal_indices` is the statistics/visualization counterpart to `lcz_get_indices`: it groups every
valid pixel (LCZ classes 1-17) by class and computes descriptive statistics (count, mean, median, std,
min/max, quartiles, and a confidence interval on the mean) per (index, LCZ class) pair. With
`plot_type="box"` it renders a precomputed-statistics Plotly box plot per index — one subplot per
index when multiple are requested (small-multiples), colored with the official LCZ palette.

In [5]:
from LCZ4py import lcz_cal_indices

stats_box = lcz_cal_indices(lcz_map_path, idx, plot_type="box")
stats_box.df.head(10)

lcz,count,mean,median,std,min,max,q25,q75,parameter,ci_lower,ci_upper,lcz_name,lcz_col,lcz_colorblind,type
i16,u64,f64,f64,f64,f64,f64,f64,f64,str,f64,f64,str,str,str,str
2,42,0.074798,0.058002,0.060319,0.009994,0.248502,0.030144,0.092912,"""NDVI""",0.056002,0.093595,"""Compact midrise""","""#D9081C""","""#D8755E""","""Built-up"""
3,2469,0.12318,0.105893,0.076542,-0.01445,0.517963,0.069951,0.158686,"""NDVI""",0.12016,0.126201,"""Compact lowrise""","""#FF0A22""","""#C98027""","""Built-up"""
4,4,0.097217,0.108519,0.071493,0.014522,0.157306,0.060411,0.156628,"""NDVI""",-0.016544,0.210977,"""Open highrise""","""#C54F1E""","""#B48C00""","""Built-up"""
5,40,0.16173,0.165016,0.08796,0.012172,0.426943,0.106079,0.203994,"""NDVI""",0.133599,0.189861,"""Open midrise""","""#FF6628""","""#989600""","""Built-up"""
6,4670,0.246287,0.236447,0.106296,-0.060494,0.616136,0.169009,0.31732,"""NDVI""",0.243237,0.249336,"""Open lowrise""","""#FF985E""","""#739F00""","""Built-up"""
8,921,0.127259,0.106529,0.087668,-0.034757,0.596998,0.061129,0.179592,"""NDVI""",0.12159,0.132928,"""Large lowrise""","""#BBBBBB""","""#00AA63""","""Built-up"""
9,13925,0.301454,0.299683,0.073721,-0.075583,0.597904,0.254354,0.348292,"""NDVI""",0.30023,0.302679,"""Sparsely built""","""#FFCBAB""","""#00AD89""","""Built-up"""
10,90,0.052924,0.048346,0.033259,0.011137,0.184022,0.026228,0.066818,"""NDVI""",0.045958,0.05989,"""Heavy Industry""","""#565656""","""#00ACAA""","""Built-up"""
11,24024,0.493326,0.510627,0.073114,-0.009621,0.67325,0.467507,0.539119,"""NDVI""",0.492401,0.494251,"""Dense trees""","""#006A18""","""#00A7C5""","""Natural"""


**How to read it:** each box is one LCZ class's pixel distribution for one index. For NDVI, expect natural classes (11-17, especially LCZ D — low plants, and LCZ A — dense trees) to sit well above urban classes (1-10), with LCZ 1-3 (compact high/mid/lowrise) showing the lowest, tightest NDVI boxes. For NDBI the pattern typically inverts: compact urban classes show higher (often positive) NDBI, natural classes show strongly negative NDBI. If a box is very wide or has extreme whiskers, check `stats_box.df` for that class's `count` — small-city extracts like Juiz de Fora can have very few pixels in some LCZ classes, making the distribution unstable. `stats_box.fig` is the Plotly figure; call `.show()` on it interactively.

## 5. Per-LCZ-class statistics: bar chart with confidence intervals

`plot_type="bar"` renders the same underlying statistics as a mean ± 95% confidence-interval bar per
class — often easier to read than a box plot when you mainly care about central tendency and whether
class means are statistically distinguishable (non-overlapping CIs is a rough, informal signal of a
real difference; the effect-size chart in the next section is the rigorous version of that
comparison).

In [6]:
stats_bar = lcz_cal_indices(lcz_map_path, idx, plot_type="bar", confidence=0.95)
stats_bar.df.select(["parameter", "lcz", "lcz_name", "mean", "ci_lower", "ci_upper"]).head(10)

parameter,lcz,lcz_name,mean,ci_lower,ci_upper
str,i16,str,f64,f64,f64
"""NDVI""",2,"""Compact midrise""",0.074798,0.056002,0.093595
"""NDVI""",3,"""Compact lowrise""",0.12318,0.12016,0.126201
"""NDVI""",4,"""Open highrise""",0.097217,-0.016544,0.210977
"""NDVI""",5,"""Open midrise""",0.16173,0.133599,0.189861
"""NDVI""",6,"""Open lowrise""",0.246287,0.243237,0.249336
"""NDVI""",8,"""Large lowrise""",0.127259,0.12159,0.132928
"""NDVI""",9,"""Sparsely built""",0.301454,0.30023,0.302679
"""NDVI""",10,"""Heavy Industry""",0.052924,0.045958,0.05989
"""NDVI""",11,"""Dense trees""",0.493326,0.492401,0.494251


The error bars are a two-tailed t-distribution confidence interval on the mean (`confidence=0.95` by default) — narrower for classes with more pixels, wider for sparsely sampled ones. Note the ponytail-flagged simplification in the source: these CIs assume independent pixels, with no spatial-autocorrelation correction, so treat them as a quick descriptive signal rather than a publication-grade inferential test.

## 6. Effect sizes: Cohen's d (built-up vs. natural)

This is the statistically rigorous version of "LCZ 2 looks less green than LCZ 6 in the boxplot."
`plot_type="effect_size"` computes Cohen's d — the standardized mean difference — between all
built-up pixels (LCZ 1-10 pooled) and all natural pixels (LCZ 11-17 pooled) for each requested index,
and ranks them in a horizontal bar chart with reference lines at the conventional thresholds
(|d| = 0.2 small, 0.5 medium, 0.8 large). The underlying table is returned separately in
`.cohens_d` so you can build custom class-vs-class comparisons (e.g. LCZ 2 vs. LCZ 6 specifically)
from the same raw pixel arrays if needed.

In [7]:
stats_effect = lcz_cal_indices(lcz_map_path, idx, plot_type="effect_size")
stats_effect.cohens_d.sort("cohens_d", descending=True)

parameter,cohens_d,magnitude,mean_urban,mean_natural,n_urban,n_natural
str,f64,str,f64,f64,i64,i64
"""NDBI""",0.917399,"""Large""",0.031343,-0.070487,22161,76131
"""MNDWI""",0.352383,"""Small""",-0.323021,-0.344129,22161,76131
"""EVI""",-1.220066,"""Large""",0.246961,0.383607,22161,76131
"""NDVI""",-1.320579,"""Large""",0.261,0.399658,22161,76131


Expect NDBI and NDVI to show the largest |d| — they are, almost by construction, the indices most diagnostic of the urban/natural split that LCZ's 1-10 vs. 11-17 grouping encodes. A large |d| (>0.8) for NDVI, for example, lets you state precisely: "built-up LCZ classes have significantly less vegetation than natural classes, with a large effect size" — with the `mean_urban`, `mean_natural`, `n_urban`, and `n_natural` columns as the receipts behind that claim. A small |d| for an index means it doesn't distinguish urban from natural land cover well in this scene (which can happen for water-family indices in an inland city with little open water, like Juiz de Fora).

## 7. Bonus: `lcz_cal_indexes`, a general zonal-statistics tool

`lcz_cal_indices` above is purpose-built for spectral index stacks. But zonal statistics by LCZ class
— "summarize this raster's pixel values per class" — is a generic operation that shows up throughout
LCZ4py: for gridded ERA5 temperature, CHIRPS precipitation, pollution rasters, and more (all covered
in notebook 07). `lcz_cal_indexes` is that general tool, built around `LCZGridResult` (the shared
return type for every `lcz_grid_*` downloader) instead of `LCZIndicesResult`.

To show it's a drop-in replacement for any single gridded variable, we reuse the NDVI band we already
computed, wrapping it in a minimal `LCZGridResult` (just `array`, `bands`, `variables`) instead of
re-downloading anything.

In [8]:
from LCZ4py._internal._lcz_grid_raster_base import LCZGridResult
from LCZ4py import lcz_cal_indexes

ndvi_i = idx.indices.index("NDVI")
ndvi_grid = LCZGridResult(array=idx.array[ndvi_i][None, ...], bands=["NDVI"], variables=["NDVI"])

ndvi_zonal = lcz_cal_indexes(lcz_map_path, ndvi_grid, variable_name="NDVI")
ndvi_zonal.df

lcz,n_pixels,mean,std,min,max,median,lcz_name,color
i16,u64,f64,f64,f64,f64,f64,str,str
2,42,0.074798,0.060319,0.009994,0.248502,0.058002,"""Compact midrise""","""#D9081C"""
3,2469,0.12318,0.076542,-0.01445,0.517963,0.105893,"""Compact lowrise""","""#FF0A22"""
4,4,0.097217,0.071493,0.014522,0.157306,0.108519,"""Open highrise""","""#C54F1E"""
5,40,0.16173,0.08796,0.012172,0.426943,0.165016,"""Open midrise""","""#FF6628"""
6,4670,0.246287,0.106296,-0.060494,0.616136,0.236447,"""Open lowrise""","""#FF985E"""
…,…,…,…,…,…,…,…,…
13,1,0.142811,null,0.142811,0.142811,0.142811,"""Bush, scrub""","""#628432"""
14,6427,0.318589,0.061952,0.051844,0.629713,0.316427,"""Low plants""","""#B5DA7F"""
15,21,0.02609,0.115546,-0.220698,0.19595,0.040646,"""Bare rock or paved""","""#000000"""


The output table (`n_pixels, mean, std, min, max, median` per LCZ class) is structurally the same shape you'd get from summarizing ERA5 temperature or CHIRPS rainfall in notebook 07 — same function, same per-class bar chart, just a different variable plugged in. Compare `ndvi_zonal.df`'s per-class means against `stats_box.df` filtered to `parameter == "NDVI"` — they should match, since both are zonal means of the same underlying NDVI band.

## Conclusion & next steps

You downloaded a Sentinel-2 scene, computed NDVI/NDBI/MNDWI/EVI, and summarized them per LCZ class
with descriptive statistics, box/bar charts, and rigorous Cohen's d effect sizes — turning "urban
classes look less green" into a quantified, defensible statement. You also saw that the same
zonal-statistics machinery (`lcz_cal_indexes`) generalizes to any gridded variable, not just spectral
indices.

**Previous:** [`04_remote_sensing_lst_pc`](./04_remote_sensing_lst_pc.en.ipynb) — land surface
temperature from GOES/Sentinel-3, and Planetary Computer basics.

**Next:** [`06_urban_canopy_parameters`](./06_urban_canopy_parameters.en.ipynb) — downloading and
processing urban canopy parameters (building height, imperviousness, vegetation fraction) from
global datasets, complementary to the spectral-index view in this notebook.